# Full Inversion — *Salmonella* Hi-C  (Colab edition)

Polymer-physics inversion of a bacterial Hi-C contact map using OpenMiChroM. Adapted from the [official tutorial](https://open-michrom.readthedocs.io/en/latest/Tutorials/Tutorial_Full_Inversion_Optimization.html).

Iteratively trains a contact-probability potential (Adam) so that an MD simulation of a 20-bead circular polymer reproduces the experimental Hi-C of *Salmonella* at 250 kb resolution.

**Inputs:** `input/salmonella_250kb.dense` (the raw experimental matrix).

**Outputs kept:** per iteration, `input/lambda_i` (potential) + `output/hic_sim_i.dense` (simulated map). 

In [ ]:
# Install before importing scientific packages. Run on a fresh Colab runtime.
import sys
import subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'openmm[cuda12]==8.6.1', 'OpenMiChroM==1.1.1',
                           'numpy', 'pandas', 'matplotlib', 'scikit-learn', 'h5py'])
else:
    print('Local run: use your existing OpenMM/OpenMiChroM environment.')

## 0. Student setup

1. Open this notebook in Colab and save your own copy.
2. Select **Runtime → Change runtime type → GPU** before running cells.
3. Run the installation cell above, then continue from top to bottom. If Colab requests a session restart, restart and rerun from the top.
4. The cell below checks CUDA by creating a small OpenMM context before any training.

The installation uses OpenMiChroM 1.1.1 and OpenMM 8.6.1 with CUDA 12. This updates the original 1.0.7 setup; newly generated results need not match the older run exactly.

**Saving:** set `SAVE_TO_DRIVE=True` to retain your results through runtime deletion (Google will request Drive access). Otherwise download the results archive at the end. Use a new `RUN_NAME` for an independent run; keep it to continue a completed sequence of iterations.

In [ ]:
import os
from pathlib import Path
from importlib.metadata import version
import openmm as mm
from openmm import unit

SAVE_TO_DRIVE = False
RUN_NAME = 'salmonella_workshop_01'  # Change for a fresh, independent run.
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.', '..')
if IN_COLAB:
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        base = Path('/content/drive/MyDrive/sbbf2026')
    else:
        base = Path('/content/sbbf2026')
    WORKDIR = base / RUN_NAME
else:
    WORKDIR = Path.cwd()
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)

SIM_PLATFORM = 'CUDA' if IN_COLAB else 'CPU'
platform = mm.Platform.getPlatformByName(SIM_PLATFORM)
system = mm.System()
system.addParticle(1.0)
integrator = mm.VerletIntegrator(0.001)
properties = {'DeviceIndex': '0', 'Precision': 'mixed'} if SIM_PLATFORM == 'CUDA' else {}
try:
    context = mm.Context(system, integrator, platform, properties)
    context.setPositions([[0, 0, 0]])
    integrator.step(1)
    print('OpenMM device check passed:', context.getPlatform().getName())
finally:
    if 'context' in globals(): del context
    del integrator
print('Working directory:', Path.cwd())
for package in ['OpenMM', 'OpenMiChroM', 'numpy', 'pandas']:
    print(package, version(package))

### Download the experimental input

No manual upload or precomputed training files are required. The download below runs **after** selecting the working directory. Existing input is reused. If the download fails, check internet access and rerun this cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize

RAW_FILE  = 'input/salmonella_250kb.dense'
NORM_FILE = 'input/salmonella_250kb_norm.dense'
SEQ_FILE  = 'input/seq_salmonella'

plt.rcParams['figure.dpi'] = 110
from urllib.request import urlopen
if not Path(RAW_FILE).exists():
    url = 'https://raw.githubusercontent.com/angmr7/full-inversion-tutorial/main/input/salmonella_250kb.dense'
    with urlopen(url, timeout=60) as response:
        raw_bytes = response.read()
    import io
    check = np.loadtxt(io.BytesIO(raw_bytes))
    assert check.ndim == 2 and check.shape[0] == check.shape[1] and check.shape[0] > 2
    Path(RAW_FILE).write_bytes(raw_bytes)
print('Input ready:', RAW_FILE)

## 1. Load and normalize the experimental Hi-C

Tutorial recipe: upper-triangle → NaN→0 → row-wise max → symmetrize → unit diagonal.  
Result lives in `[0, 1]` and is the contact-probability target for the optimizer.

In [ ]:
M_raw = np.loadtxt(RAW_FILE)
N = M_raw.shape[0]
print(f'Raw shape: {M_raw.shape}, range [{M_raw.min():.0f}, {M_raw.max():.0f}]')

r = np.triu(M_raw, k=1)
r[np.isnan(r)] = 0.0
r = normalize(r, axis=1, norm='max')
M_norm = r + r.T + np.diag(np.ones(N))

np.savetxt(NORM_FILE, M_norm)
print(f'Normalized -> {NORM_FILE}, range [{M_norm.min():.3f}, {M_norm.max():.3f}]')

fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

im0 = ax[0].imshow(np.log10(M_raw + 1), cmap='Reds')
ax[0].set_title('Raw (log10)')
fig.colorbar(im0, ax=ax[0], label='log10(contact + 1)')

im1 = ax[1].imshow(M_norm, cmap='Reds', vmin=0, vmax=1)
ax[1].set_title('Normalized')
fig.colorbar(im1, ax=ax[1], label='Normalized contact')

plt.show()

## 2. Sequence file and `lambda_0`

Trivial in-memory replacements for `2_create_seq.sh` and `3_create_lambda_0.py`.

In [ ]:
bead_names = [f'Bead{i+1}' for i in range(N)]
with open(SEQ_FILE, 'w') as f:
    for i, name in enumerate(bead_names, start=1):
        f.write(f'{i} {name}\n')
print(f'{SEQ_FILE}: {N} beads')

lambda_0 = pd.DataFrame(np.zeros((N, N)), columns=bead_names, index=bead_names)
if not Path('input/lambda_0').exists():
    lambda_0.to_csv('input/lambda_0', sep=' ', index=False)
    print('lambda_0 written (all zeros)')
else:
    print('Existing lambda_0 preserved.')

## 3. One iteration on a single GPU

Replicas run sequentially and accumulate contact samples in the same optimizer. Each iteration simulates `lambda_i`, writes the simulated map, then produces `lambda_{i+1}`. Adam momentum and velocity are saved in `Adam/` and must be retained alongside the lambdas.

Start with a short trial (`MAX_ITER=2`, `N_BLOCKS=10`, `REPLICAS=1`) to check installation and timing. Those settings only test the workflow; they are not sufficient for scientific convergence. Use a new `RUN_NAME` when changing sampling or learning parameters for an independent run.

In [ ]:
from OpenMiChroM.ChromDynamics import MiChroM
from OpenMiChroM.Optimization import AdamTraining

# --- knobs (parameters for the simulations)                  ---------------
MU, RC, ETA   = 3.22, 1.78, 0.03
TEMPERATURE   = 1.0
TIME_STEP     = 0.01
BLOCK         = 1000      # MD steps per Pi sample
N_BLOCKS      = 300       # samples per replica; measure runtime on your assigned GPU
REPLICAS      = 2         # sequential replicas per iteration
EQUIL_STEPS   = 1000      # quick equilibration
GPU_ID        = '0'       # OpenMiChroM expects a string here
# --------------------------------------------------------------------------

def run_replica(replica_id, seq_file, lambda_file, opt, output_dir):
    sim = MiChroM(name=f'rep{replica_id}', temperature=TEMPERATURE, timeStep=TIME_STEP)
    sim.setup(platform=SIM_PLATFORM, gpu=GPU_ID, integrator='langevin',
              precision='mixed', deviceIndex=GPU_ID)
    sim.saveFolder(output_dir)
    sim.loadStructure(sim.createSpringSpiral(ChromSeq=seq_file, isRing=True), center=True)
    sim.addFENEBonds(kFb=30.0)
    sim.addAngles(kA=2.0)
    sim.addRepulsiveSoftCore(eCut=4.0)
    sim.addCustomTypes(mu=MU, rc=RC, TypesTable=lambda_file)
    sim.addFlatBottomHarmonic(kR=5e-3, nRad=8.0)
    sim.createSimulation()
    actual_platform = sim.simulation.context.getPlatform().getName()
    if IN_COLAB and actual_platform != 'CUDA':
        raise RuntimeError(f'Expected CUDA, but simulation selected {actual_platform}.')
    sim.run(nsteps=EQUIL_STEPS, report=False, interval=1000)
    sim.removeFlatBottomHarmonic()
    for _ in range(N_BLOCKS):
        sim.simulation.step(BLOCK)
        opt.probCalc(sim.getPositions())
    return opt

def run_iteration(iteration):
    lambda_in  = f'input/lambda_{iteration}'
    lambda_out = f'input/lambda_{iteration + 1}'
    out_dir    = f'output/iteration_{iteration}'
    os.makedirs(out_dir, exist_ok=True)

    opt = AdamTraining(mu=MU, rc=RC, eta=ETA, it=iteration + 1, updateNeeded=True, update_storagePath="Adam")
    opt.getHiCexp(HiC=NORM_FILE, norm=False, cutoff_low=0.001, neighbors=1)

    for rep_id in range(1, REPLICAS + 1):
        print(f'  iter {iteration} | replica {rep_id}/{REPLICAS} ...', flush=True)
        opt = run_replica(rep_id, SEQ_FILE, lambda_in, opt, out_dir)

    M_target = np.loadtxt(NORM_FILE)
    zero_pts = [tuple(p) for p in np.argwhere(M_target == 0)]
    lambda_new = opt.getLamb(Lambdas=lambda_in, fixedPoints=zero_pts)

    hic_sim = opt.Pi / opt.NFrames
    np.savetxt(f'{out_dir}/hic_sim_{iteration}.dense', hic_sim)
    lambda_new.to_csv(lambda_out, sep=' ', index=False)

    # cleanup: remove every intermediate file the C++ side may have dropped
    for ext in ('.gro', '.cndb', '.h5'):
        for f in os.listdir(out_dir):
            if f.endswith(ext):
                try: os.remove(os.path.join(out_dir, f))
                except OSError: pass

    return hic_sim, opt.error

## 4. Run the full inversion loop

The full configuration below requests **80 iterations**. First time one or two iterations to estimate classroom runtime; GPU speed and allocation vary.

Completed iterations are skipped only when their completion marker, map, and updated lambda all exist. Keep the same sampling settings and run folder to extend a run. If a runtime disconnects during an iteration, the pending marker prevents an unsafe restart with potentially advanced Adam state. For a fresh workshop run, choose a new `RUN_NAME` and rerun from setup. Do not delete the marker to bypass this check.

In [ ]:
import time
import json

MAX_ITER = 80  # Set to 2 for your initial trial, then extend if desired.
errors = []
pending = Path('output/pending_iteration.txt')
if pending.exists():
    raise RuntimeError('An iteration was interrupted. Choose a new RUN_NAME for a fresh run; '
                       'the current Adam state may not match the last completed lambda.')
settings = dict(MU=MU, RC=RC, ETA=ETA, TEMPERATURE=TEMPERATURE, TIME_STEP=TIME_STEP,
                BLOCK=BLOCK, N_BLOCKS=N_BLOCKS, REPLICAS=REPLICAS, EQUIL_STEPS=EQUIL_STEPS,
                OpenMiChroM=version('OpenMiChroM'), OpenMM=version('OpenMM'))
settings_path = Path('output/training_settings.json')
if settings_path.exists() and json.loads(settings_path.read_text()) != settings:
    raise RuntimeError('Training settings differ from this saved run. Choose a new RUN_NAME.')
settings_path.write_text(json.dumps(settings, indent=2))
for it in range(MAX_ITER):
    marker = Path(f'output/iteration_{it}/completed.json')
    next_lambda = Path(f'input/lambda_{it + 1}')
    contact_map = Path(f'output/iteration_{it}/hic_sim_{it}.dense')
    if marker.exists() and next_lambda.exists() and contact_map.exists():
        print(f'iter {it} already complete, skipping')
        continue
    if marker.exists() or next_lambda.exists() or contact_map.exists():
        raise RuntimeError(f'Incomplete or older results at iteration {it}; choose a new RUN_NAME.')
    if it > 0 and not all(Path('Adam', name).exists() for name in
                          ['iteration.txt', 'moment.txt', 'velocity.txt']):
        raise RuntimeError('Missing Adam checkpoint; choose a new RUN_NAME for a fresh run.')
    pending.write_text(str(it))
    t0 = time.time()
    hic_sim, err = run_iteration(it)
    dt = time.time() - t0
    marker.write_text(json.dumps(dict(iteration=it, error=float(err), seconds=dt)))
    pending.unlink()
    errors.append((it, err))
    print(f'iter {it} | error = {err:.4f} | {dt/60:.1f} min')

## 5. Convergence and scaling 

Three checks:

1. **Hi-C** experimental vs simulated (final iteration).
2. **Contact decay** in *circular* distance — slope and shape.
3. **Full scaling** in *linear* distance — should show the U-shape expected for a circular polymer (a consistency check, not proof of topology).

In [ ]:
# Pick the last completed iteration
done = sorted(int(d.split('_')[1]) for d in os.listdir('output') if d.startswith('iteration_'))
last = done[-1] if done else None
assert last is not None, 'No iterations completed yet.'

E = np.loadtxt(NORM_FILE)
S = np.loadtxt(f'output/iteration_{last}/hic_sim_{last}.dense')
n = E.shape[0]

def decay(M, circular=True):
    i, j = np.indices(M.shape)
    d = np.minimum(np.abs(i - j), n - np.abs(i - j)) if circular else np.abs(i - j)
    md = d.max()
    return np.arange(md + 1), np.array([M[d == k].mean() for k in range(md + 1)])

dE_c, mE_c = decay(E, True);  dS_c, mS_c = decay(S, True)
dE_l, mE_l = decay(E, False); dS_l, mS_l = decay(S, False)
sE = np.polyfit(np.log10(dE_c[1:]), np.log10(mE_c[1:]), 1)[0]
sS = np.polyfit(np.log10(dS_c[1:]), np.log10(mS_c[1:]), 1)[0]
rho = np.corrcoef(E[np.triu_indices(n, 1)], S[np.triu_indices(n, 1)])[0, 1]

fig, ax = plt.subplots(2, 2, figsize=(11, 9))

im = ax[0, 0].imshow(E, cmap='Reds', vmin=0, vmax=1); ax[0, 0].set_title('Experimental'); plt.colorbar(im, ax=ax[0, 0])
im = ax[0, 1].imshow(S, cmap='Reds', vmin=0, vmax=1); ax[0, 1].set_title(f'Simulated  (iter {last})'); plt.colorbar(im, ax=ax[0, 1])

ax[1, 0].loglog(dE_c[1:], mE_c[1:], 'o-', color='steelblue', label=f'Exp  slope {sE:+.2f}')
ax[1, 0].loglog(dS_c[1:], mS_c[1:], 's--', color='darkred',   label=f'Sim  slope {sS:+.2f}')
ax[1, 0].set_xlabel('Circular distance (bins)'); ax[1, 0].set_ylabel('Mean contact')
ax[1, 0].set_title(f'Decay  (Pearson off-diag = {rho:.3f})'); ax[1, 0].legend(); ax[1, 0].grid(alpha=0.3, which='both')

ax[1, 1].plot(dE_l[1:], mE_l[1:], 'o-', color='steelblue', label='Exp')
ax[1, 1].plot(dS_l[1:], mS_l[1:], 's--', color='darkred',  label='Sim')
ax[1, 1].set_yscale('log')
ax[1, 1].axvline(n // 2, color='gray', ls=':', label=f'N/2 = {n // 2}')
ax[1, 1].set_xlabel('Linear distance |i - j|'); ax[1, 1].set_ylabel('Mean contact')
ax[1, 1].set_title('Full scaling — Circular-topology consistency check'); ax[1, 1].legend(); ax[1, 1].grid(alpha=0.3, which='both')

plt.tight_layout(); plt.show()
print(f'\nFinal slope error  : {abs(sS - sE):.3f}')
print(f'Off-diag Pearson   : {rho:.3f}')

### All iterations: simulated Hi-C maps and contact decay

The grid uses one shared color scale so maps can be compared across iterations. The decay panel overlays each saved simulation with the experimental target; color encodes the simulation iteration. Only iterations with a saved `hic_sim_i.dense` file are included.


In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

analysis_target = np.loadtxt(NORM_FILE)
analysis_n = analysis_target.shape[0]
map_files = []
for f in Path('output').glob('iteration_*/hic_sim_*.dense'):
    try:
        iteration = int(f.stem.split('_')[-1])
    except ValueError:
        continue
    if f.parent.name == f'iteration_{iteration}':
        matrix = np.loadtxt(f)
        if matrix.shape == analysis_target.shape:
            map_files.append((iteration, matrix))
map_files.sort(key=lambda item: item[0])
assert map_files, 'No simulated Hi-C maps found in output/iteration_*/.'

# Shared color limits make the heatmaps comparable across all iterations.
color_max = max(float(np.nanmax(analysis_target)), *(float(np.nanmax(m)) for _, m in map_files))
columns = min(5, len(map_files))
rows = math.ceil(len(map_files) / columns)
fig, axes = plt.subplots(rows, columns, figsize=(3.0 * columns, 2.8 * rows),
                         squeeze=False, constrained_layout=True)
for ax, (iteration, matrix) in zip(axes.ravel(), map_files):
    im = ax.imshow(matrix, cmap='Reds', vmin=0, vmax=color_max, origin='lower')
    ax.set_title(f'Iteration {iteration}')
    ax.set_xlabel('Bead index')
    ax.set_ylabel('Bead index')
for ax in axes.ravel()[len(map_files):]:
    ax.set_visible(False)
fig.colorbar(im, ax=axes.ravel().tolist(), label='Contact probability', shrink=0.85)
plt.show()

# Use the same circular genomic-distance definition as the convergence plot above.
i_idx, j_idx = np.indices(analysis_target.shape)
circular_distance = np.minimum(np.abs(i_idx - j_idx), analysis_n - np.abs(i_idx - j_idx))
max_distance = int(circular_distance.max())
distances = np.arange(max_distance + 1)
mean_exp = np.array([analysis_target[circular_distance == d].mean() for d in distances])
iterations = [iteration for iteration, _ in map_files]
cmap = plt.get_cmap('viridis')
norm = Normalize(vmin=min(iterations), vmax=max(iterations) if max(iterations) > min(iterations) else min(iterations) + 1)
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.loglog(distances[1:], mean_exp[1:], color='black', lw=2.5, label='Experimental')
for iteration, matrix in map_files:
    mean_sim = np.array([matrix[circular_distance == d].mean() for d in distances])
    ax.loglog(distances[1:], mean_sim[1:], color=cmap(norm(iteration)), alpha=0.85, lw=1.5)
ax.set(xlabel='Circular distance (bins)', ylabel='Mean contact probability',
       title='Contact decay across all saved iterations')
ax.grid(alpha=0.3, which='both')
ax.legend()
if len(iterations) > 1:
    fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, label='Simulation iteration')
plt.show()
print(f'Plotted {len(map_files)} simulated maps: iterations {iterations}')


## 6. Load saved training history (no MD required)

Run this section independently after importing NumPy, pandas, and Matplotlib. It reads the saved files without changing them. Here **iteration i** denotes a simulation using `lambda_i`; that simulation produces the update `lambda_{i+1}`. Thus the last saved lambda has not necessarily been simulated yet.

Lambda files store pair coefficients in the **upper triangle**; reflect that triangle for display rather than averaging it with the zero-filled lower triangle. Analysis below uses unique pairs with circular separation greater than one bin and positive experimental target. This is an explicitly chosen comparison mask, not a reconstruction of every internal optimizer filter. Neighbor pairs, including the ring closure, are excluded from these summaries.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

analysis_root = Path('.')  # Folder containing input/ and output/
target = np.loadtxt(analysis_root / 'input/salmonella_250kb_norm.dense')
assert target.ndim == 2 and target.shape[0] == target.shape[1]
n_analysis = target.shape[0]
pair_i, pair_j = np.triu_indices(n_analysis, k=1)
pair_distance = np.minimum(pair_j - pair_i, n_analysis - (pair_j - pair_i))
keep = (pair_distance > 1) & np.isfinite(target[pair_i, pair_j]) & (target[pair_i, pair_j] > 0)
pi, pj = pair_i[keep], pair_j[keep]
assert len(pi), 'No pairs available for analysis.'

def numbered_files(folder, pattern):
    return sorted((int(m.group(1)), path) for path in folder.glob('*')
                  if (m := re.fullmatch(pattern, path.name)) and path.is_file())

lambda_files = numbered_files(analysis_root / 'input', r'lambda_(\d+)')
assert lambda_files, 'No saved lambda tables found.'
lambda_iterations = np.array([it for it, _ in lambda_files])
lambda_tables = []
for it, path in lambda_files:
    table = pd.read_csv(path, sep=r'\s+').to_numpy(dtype=float)
    assert table.shape == target.shape and np.isfinite(table).all(), f'Invalid lambda table: {path}'
    lambda_tables.append(table)
lambda_tables = np.stack(lambda_tables)
lambda_pairs = lambda_tables[:, pi, pj]

saved_maps = {}
for path in (analysis_root / 'output').glob('iteration_*/hic_sim_*.dense'):
    match = re.fullmatch(r'hic_sim_(\d+)\.dense', path.name)
    if match and path.parent.name == f'iteration_{int(match.group(1))}':
        matrix = np.loadtxt(path)
        assert matrix.shape == target.shape and np.isfinite(matrix).all(), f'Invalid map: {path}'
        saved_maps[int(match.group(1))] = matrix
print(f'{n_analysis} beads; {len(pi)} comparison pairs')
print(f'Lambda iterations: {lambda_iterations.tolist()}')
print(f'Simulated iterations: {sorted(saved_maps)}')

## 7. How does the trained force field evolve?

Shared color limits make the snapshots directly comparable. The trajectory panel shows a fixed, reproducible sample of pairs, plus the median and central 80% range across all comparison pairs. Update sizes measure parameter movement; small updates alone do not establish a good fit or adequate sampling.

These are contact-potential coefficients, not forces themselves. Their physical effect also depends on the contact function and the other polymer energy terms. Values are shown in the units saved by the optimizer.

In [ ]:
snapshot_indices = sorted(set([0, len(lambda_iterations) // 2, len(lambda_iterations) - 1]))
color_limit = max(float(np.max(np.abs(np.triu(lambda_tables, k=1)))), 1e-12)
fig, axes = plt.subplots(1, len(snapshot_indices), figsize=(4 * len(snapshot_indices), 3.8), squeeze=False, constrained_layout=True)
for ax, k in zip(axes[0], snapshot_indices):
    upper = np.triu(lambda_tables[k], k=1)
    display = upper + upper.T
    im = ax.imshow(display, cmap='RdBu_r', vmin=-color_limit, vmax=color_limit, origin='lower')
    ax.set(title=f'lambda_{lambda_iterations[k]}', xlabel='Bead index (0-based)', ylabel='Bead index (0-based)')
fig.colorbar(im, ax=axes.ravel().tolist(), label='Contact-potential coefficient')
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
chosen = np.random.default_rng(7).choice(len(pi), size=min(12, len(pi)), replace=False)
ax[0].plot(lambda_iterations, lambda_pairs[:, chosen], alpha=0.35, lw=1)
q10, median, q90 = np.percentile(lambda_pairs, [10, 50, 90], axis=1)
# ax[0].fill_between(lambda_iterations, q10, q90, color='steelblue', alpha=0.2, label='10–90% of pairs')
ax[0].plot(lambda_iterations, median, color='black', lw=2, label='Median')
ax[0].set(xlabel='Lambda iteration', ylabel='Coefficient', title='Evolution of pair interactions')
ax[0].legend()
if len(lambda_iterations) > 1:
    # Divide by iteration gap if intermediate saved tables are missing.
    delta = np.diff(lambda_pairs, axis=0) / np.diff(lambda_iterations)[:, None]
    ax[1].plot(lambda_iterations[1:], np.sqrt(np.mean(delta**2, axis=1)), 'o-', label='RMS change')
    ax[1].plot(lambda_iterations[1:], np.max(np.abs(delta), axis=1), 's--', label='Largest absolute change')
    ax[1].legend()
else:
    ax[1].text(0.5, 0.5, 'At least two lambda tables are needed.', ha='center', transform=ax[1].transAxes)
ax[1].set(xlabel='Destination lambda iteration', ylabel='Coefficient change per iteration', title='Size of parameter updates')
for a in ax: a.grid(alpha=0.25)
plt.show()

## 8. Does the contact-map fit improve?

Recompute RMSE, mean absolute error, and Pearson correlation from the saved maps using the same pair mask at every iteration. These are descriptive fit metrics, not the optimizer's internal loss. Correlation captures pattern similarity but can remain high despite a systematic amplitude error, so inspect it alongside RMSE and residuals. These are training-set diagnostics, not independent validation.

In [ ]:
def safe_pearson(x, y):
    return float(np.corrcoef(x, y)[0, 1]) if len(x) > 1 and np.std(x) > 0 and np.std(y) > 0 else np.nan

expected_pairs = target[pi, pj]
fit_rows = []
for iteration, matrix in sorted(saved_maps.items()):
    simulated_pairs = matrix[pi, pj]
    residual = simulated_pairs - expected_pairs
    fit_rows.append({'iteration': iteration, 'RMSE': np.sqrt(np.mean(residual**2)),
                     'MAE': np.mean(np.abs(residual)), 'Pearson': safe_pearson(expected_pairs, simulated_pairs)})
fit_history = pd.DataFrame(fit_rows, columns=['iteration', 'RMSE', 'MAE', 'Pearson'])
if fit_history.empty:
    print('No saved simulated maps found; run the training loop to evaluate fit.')
else:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for metric in ['RMSE', 'MAE']:
        ax[0].plot(fit_history['iteration'], fit_history[metric], 'o-', label=metric)
    ax[0].set(ylabel='Contact-probability error', title='Fit error'); ax[0].legend()
    ax[1].plot(fit_history['iteration'], fit_history['Pearson'], 'o-', color='darkred')
    ax[1].set(ylabel='Pearson correlation', ylim=(-1.05, 1.05), title='Pattern agreement')
    for a in ax: a.set_xlabel('Simulation iteration (uses lambda_i)'); a.grid(alpha=0.25)
    plt.show()
    print(fit_history.round(5).to_string(index=False))

## 9. Where does the final simulation disagree?

Positive residuals mean excess simulated contact; negative residuals mean missing contact. The scatter plot uses the same unique comparison pairs as the convergence curves. The distance panel averages residuals by circular genomic separation (250 kb per bin), revealing distance-dependent bias hidden by a single global score.

In [ ]:
if saved_maps:
    final_iteration = max(saved_maps)
    final_map = saved_maps[final_iteration]
    difference = final_map - target
    # Mask all entries outside the selected unique pairs and their reflections.
    residual_display = np.full_like(target, np.nan, dtype=float)
    residual_display[pi, pj] = difference[pi, pj]
    residual_display[pj, pi] = difference[pi, pj]
    limit = max(float(np.max(np.abs(difference[pi, pj]))), 1e-12)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    im = ax[0].imshow(residual_display, cmap='RdBu_r', vmin=-limit, vmax=limit, origin='lower')
    ax[0].set(title=f'Simulation − target (iter {final_iteration})', xlabel='Bead index (0-based)', ylabel='Bead index (0-based)')
    fig.colorbar(im, ax=ax[0], label='Contact-probability residual')
    ax[1].scatter(expected_pairs, final_map[pi, pj], s=18, alpha=0.6)
    bound = max(float(expected_pairs.max()), float(final_map[pi, pj].max()), 1e-12)
    ax[1].plot([0, bound], [0, bound], 'k--', lw=1)
    ax[1].set(xlabel='Experimental contact', ylabel='Simulated contact', title='Pairwise agreement', aspect='equal')
    distances = np.minimum(pj - pi, n_analysis - (pj - pi))
    separations = np.unique(distances)
    distance_bias = [difference[pi, pj][distances == d].mean() for d in separations]
    ax[2].plot(separations * 250, distance_bias, 'o-')
    ax[2].axhline(0, color='black', ls='--', lw=1)
    ax[2].set(xlabel='Circular genomic separation (kb)', ylabel='Mean contact residual', title='Distance-dependent bias')
    plt.show()
else:
    print('No simulated maps available for residual analysis.')

### Further workshop extensions

- **Sampling reliability:** save separate replica contact maps and compare their spread before pooling. Current pooled maps cannot supply independent replica uncertainty.
- **Structure and equilibration:** retain trajectories or per-block radius of gyration and energy to examine equilibration and structural variability. These cannot be recovered from contact maps alone.
- **Sensitivity:** compare learning rates or longer sampling runs using both map-fit metrics and lambda changes. Similar maps can arise from different parameter sets.

## 10. Fixed-lambda validation: save structure and equilibration data

Training changes the potential between iterations. To assess relaxation and structural sampling, keep the latest saved lambda **fixed** in a separate validation run. This does not change the FI outputs. Run the configuration and helper cells below, then explicitly set `RUN_VALIDATION = True` to launch MD after installing OpenMiChroM.

Each replica saves coordinates and total potential energy during confined relaxation, post-release burn-in, and production. Only production contributes to the structural distributions and contact maps below. The default durations are starting points, **not established equilibration times**: inspect the traces, compare time windows, and lengthen burn-in/production as needed. Independent velocity seeds help compare replicas, but a shared starting geometry and short runs can still hide slow modes.

Coordinates and distances use OpenMiChroM simulation length units; no conversion to physical nanometers is inferred. Time is reported in MD steps. Energy is the raw OpenMM potential energy in kJ/mol, not divided by bead count or temperature. No uncertainty interval is inferred from correlated trajectory frames.

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VALIDATION_DIR = Path('output/structure_validation')
VALIDATION_PLATFORM = 'OpenCL'  # Change to 'CUDA' or 'OpenCL' when available.
VALIDATION_REPLICAS = 2
VALIDATION_STRIDE = 1000
CONFINED_BLOCKS = 10
BURNIN_BLOCKS = 100
PRODUCTION_BLOCKS = 300
VALIDATION_DT = 0.01
VALIDATION_TEMPERATURE = 1.0
VALIDATION_MU, VALIDATION_RC = 3.22, 1.78
RUN_VALIDATION = True

candidates = [(int(m.group(1)), p) for p in Path('input').glob('lambda_*')
              if (m := re.fullmatch(r'lambda_(\d+)', p.name))]
assert candidates, 'No trained lambda files found.'
VALIDATION_ITERATION, VALIDATION_LAMBDA = max(candidates)
print(f'Fixed force field: {VALIDATION_LAMBDA}; platform: {VALIDATION_PLATFORM}')

def structural_observables(xyz):
    """Rg and relative shape anisotropy from the gyration tensor."""
    centered = np.asarray(xyz, dtype=float) - np.mean(xyz, axis=0)
    eigenvalues = np.linalg.eigvalsh(centered.T @ centered / len(centered))
    rg2 = eigenvalues.sum()
    anisotropy = 1.5 * np.sum(eigenvalues**2) / rg2**2 - 0.5 if rg2 > 0 else 0.0
    return float(np.sqrt(rg2)), float(np.clip(anisotropy, 0, 1))

def contact_frames(xyz, mu, rc):
    distances = np.linalg.norm(xyz[:, :, None, :] - xyz[:, None, :, :], axis=-1)
    return 0.5 * (1 + np.tanh(mu * (rc - distances)))

def run_structure_validation():
    from OpenMiChroM.ChromDynamics import MiChroM
    from openmm import unit
    import hashlib
    assert VALIDATION_REPLICAS >= 2
    assert min(CONFINED_BLOCKS, BURNIN_BLOCKS, PRODUCTION_BLOCKS, VALIDATION_STRIDE) > 0
    # Fail before any MD if this destination already holds saved validation data.
    if VALIDATION_DIR.exists() and any(VALIDATION_DIR.iterdir()):
        raise FileExistsError(f'{VALIDATION_DIR} is not empty. Choose a new VALIDATION_DIR for another run.')
    VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
    config = dict(lambda_iteration=VALIDATION_ITERATION, lambda_file=str(VALIDATION_LAMBDA),
                  lambda_sha256=hashlib.sha256(VALIDATION_LAMBDA.read_bytes()).hexdigest(),
                  platform=VALIDATION_PLATFORM, replicas=VALIDATION_REPLICAS,
                  stride=VALIDATION_STRIDE, confined_blocks=CONFINED_BLOCKS,
                  burnin_blocks=BURNIN_BLOCKS, production_blocks=PRODUCTION_BLOCKS,
                  dt=VALIDATION_DT, temperature=VALIDATION_TEMPERATURE,
                  mu=VALIDATION_MU, rc=VALIDATION_RC)
    (VALIDATION_DIR / 'config.json').write_text(json.dumps(config, indent=2))
    for replica in range(1, VALIDATION_REPLICAS + 1):
        seed = 2026 + replica
        sim = MiChroM(name=f'validation_{replica}', temperature=VALIDATION_TEMPERATURE, timeStep=VALIDATION_DT)
        sim.setup(platform=VALIDATION_PLATFORM, integrator='langevin', precision='single', deviceIndex='0')
        replica_dir = VALIDATION_DIR / f'replica_{replica}'
        replica_dir.mkdir()
        sim.saveFolder(str(replica_dir))
        sim.loadStructure(sim.createSpringSpiral(ChromSeq='input/seq_salmonella', isRing=True), center=True)
        sim.addFENEBonds(kFb=30.0)
        sim.addAngles(kA=2.0)
        sim.addRepulsiveSoftCore(eCut=4.0)
        sim.addCustomTypes(mu=VALIDATION_MU, rc=VALIDATION_RC, TypesTable=str(VALIDATION_LAMBDA))
        sim.addFlatBottomHarmonic(kR=5e-3, nRad=8.0)
        sim.integrator.setRandomNumberSeed(seed)
        sim.createSimulation()
        sim.context.setVelocitiesToTemperature(sim.temperature, seed)
        frames, rows, phases, steps = [], [], [], []
        step = 0
        for phase, blocks in [('confined', CONFINED_BLOCKS), ('burn-in', BURNIN_BLOCKS), ('production', PRODUCTION_BLOCKS)]:
            if phase == 'burn-in':
                sim.removeFlatBottomHarmonic()
            for _ in range(blocks):
                sim.simulation.step(VALIDATION_STRIDE)
                step += VALIDATION_STRIDE
                xyz = np.array(sim.getPositions(), dtype=float)
                energy = sim.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
                if not (np.isfinite(xyz).all() and np.isfinite(energy)):
                    raise RuntimeError(f'Nonfinite state in replica {replica}, step {step}')
                rg, anisotropy = structural_observables(xyz)
                frames.append(xyz); phases.append(phase); steps.append(step)
                rows.append(dict(step=step, phase=phase, rg=rg, anisotropy=anisotropy, potential_energy=energy))
            # Checkpoint each completed phase, so interrupted runs retain completed phases.
            np.savez_compressed(replica_dir / 'trajectory.npz', xyz=np.asarray(frames),
                                phase=np.asarray(phases), step=np.asarray(steps))
            pd.DataFrame(rows).to_csv(replica_dir / 'observables.csv', index=False)
        print(f'Replica {replica}: {len(frames)} saved frames (seed {seed})')
        del sim

if RUN_VALIDATION:
    run_structure_validation()
else:
    print('MD not launched. Set RUN_VALIDATION = True above to generate validation data.')

## 11. Relaxation traces and production distributions

The first dashed line marks confinement removal; the second marks the chosen production start. The energy discontinuity at removal reflects a change of potential and is not, by itself, instability. Running averages are visual guides, not evidence of independent sampling. Relative shape anisotropy ranges from 0 (isotropic gyration tensor) to 1 (rod-like).

Look for drift after burn-in and differences between replicas. Agreement in these observables is necessary evidence to consider, but does not prove equilibration.

In [ ]:
validation_runs = []
validation_config = None
if (VALIDATION_DIR / 'config.json').exists():
    validation_config = json.loads((VALIDATION_DIR / 'config.json').read_text())
for file in sorted(VALIDATION_DIR.glob('replica_*/trajectory.npz')):
    with np.load(file, allow_pickle=False) as data:
        xyz, phases, steps = data['xyz'], data['phase'], data['step']
    assert xyz.ndim == 3 and xyz.shape[2] == 3 and np.isfinite(xyz).all(), f'Invalid trajectory: {file}'
    table = pd.read_csv(file.parent / 'observables.csv')
    assert len(table) == len(xyz) and np.array_equal(table['step'].to_numpy(), steps)
    assert np.array_equal(table['phase'].to_numpy(), phases)
    production = phases == 'production'
    if production.sum() < 4:
        print(f'Skipping incomplete production: {file.parent.name}')
        continue
    validation_runs.append(dict(name=file.parent.name, xyz=xyz, phases=phases, steps=steps, table=table, production=production))

if not validation_runs:
    print('No completed validation trajectories yet. Generate them with section 10; existing FI maps contain no coordinates.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
    fields = [('rg', 'Radius of gyration'), ('potential_energy', 'Potential energy (kJ/mol)'), ('anisotropy', 'Relative shape anisotropy')]
    summary = []
    for run in validation_runs:
        table, prod = run['table'], run['production']
        for col, (field, label) in enumerate(fields):
            line, = axes[0, col].plot(table['step'], table[field], alpha=0.3, lw=0.7)
            # Restart smoothing at phase boundaries to avoid smearing the release jump.
            smooth = table.groupby('phase', sort=False)[field].transform(lambda x: x.rolling(10, min_periods=1).mean())
            axes[0, col].plot(table['step'], smooth, color=line.get_color(), label=run['name'])
            for phase in ['burn-in', 'production']:
                selected = table.loc[table.phase == phase, 'step']
                if len(selected): axes[0, col].axvline(selected.iloc[0], color='gray', ls='--', alpha=0.4)
            values = table.loc[prod, field]
            axes[1, col].hist(values, bins=20, density=True, histtype='step', label=run['name'])
            summary.append(dict(replica=run['name'], observable=field, mean=values.mean(), std=values.std()))
            axes[0, col].set(xlabel='MD step', ylabel=label)
            axes[1, col].set(xlabel=label, ylabel='Production density')
    axes[0, 0].legend(); axes[1, 0].legend()
    plt.show()
    print(pd.DataFrame(summary).round(4).to_string(index=False))

## 12. Contact-map stability and replica agreement

Split production into four contiguous windows and compare each window's contact map with the pooled production map of that replica. This reference overlaps each window, so this is a descriptive drift diagnostic, not independent validation or an error bar. Also compare the first and last windows directly and compare production maps between replicas.

Use the optimizer's smooth contact function, with `mu` and `rc` read from the saved validation configuration. Metrics use unique positive-target pairs with circular separation greater than one bin, matching section 6's analysis mask. Longer runs and additional replicas are needed if time windows or replicas disagree.

In [ ]:
production_maps = {}
if validation_runs:
    assert validation_config is not None, 'Missing config.json; contact function parameters are required.'
    target_structure = np.loadtxt('input/salmonella_250kb_norm.dense')
    ni = target_structure.shape[0]
    ii, jj = np.triu_indices(ni, 1)
    selected = (np.minimum(jj - ii, ni - (jj - ii)) > 1) & np.isfinite(target_structure[ii, jj]) & (target_structure[ii, jj] > 0)
    ii, jj = ii[selected], jj[selected]
    assert len(ii), 'No eligible pairs.'
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    stability_rows = []
    for run in validation_runs:
        coords = run['xyz'][run['production']]
        assert coords.shape[1] == ni, 'Target/trajectory bead count mismatch.'
        windows = np.array_split(coords, 4)
        maps = [contact_frames(w, validation_config['mu'], validation_config['rc']).mean(axis=0) for w in windows]
        pooled = np.average(maps, axis=0, weights=[len(w) for w in windows])
        production_maps[run['name']] = pooled
        rmses = [np.sqrt(np.mean((m[ii, jj] - pooled[ii, jj])**2)) for m in maps]
        axes[0].plot(range(1, 5), rmses, 'o-', label=run['name'])
        stability_rows.append(dict(replica=run['name'], first_last_RMSE=np.sqrt(np.mean((maps[0][ii,jj]-maps[-1][ii,jj])**2)),
                                   target_RMSE=np.sqrt(np.mean((pooled[ii,jj]-target_structure[ii,jj])**2))))
    names = list(production_maps)
    between = np.array([[np.sqrt(np.mean((production_maps[a][ii,jj]-production_maps[b][ii,jj])**2)) for b in names] for a in names])
    im = axes[1].imshow(between, cmap='viridis', vmin=0)
    axes[1].set_xticks(range(len(names)), names, rotation=30); axes[1].set_yticks(range(len(names)), names)
    axes[1].set_title('Between-replica contact-map RMSE'); fig.colorbar(im, ax=axes[1])
    axes[0].set(xlabel='Production window', ylabel='RMSE from pooled replica map', xticks=[1,2,3,4]); axes[0].legend()
    plt.show()
    print(pd.DataFrame(stability_rows).round(5).to_string(index=False))
else:
    print('Contact stability requires validation trajectories.')

## 13. Structures and spatial scaling

Snapshots are evenly spaced through production, not selected as typical or most probable structures. Bead color follows genomic order; the closing bond is explicitly drawn. All snapshots share a centered coordinate scale; orientations are not aligned.

Here `R(s)` is the root-mean-square spatial distance at circular genomic separation `s`, averaged over bead pairs and production frames. Distances are in simulation units; genomic separation uses 250 kb per bin. Curves represent individual replicas without treating correlated frames as independent samples.

In [ ]:
if validation_runs:
    count = len(validation_runs)
    fig = plt.figure(figsize=(12, 3.8 * count), constrained_layout=True)
    snapshots = [run['xyz'][run['production']][np.linspace(0, run['production'].sum()-1, 3, dtype=int)] for run in validation_runs]
    centered = [frames - frames.mean(axis=1, keepdims=True) for frames in snapshots]
    extent = max(float(np.abs(frames).max()) for frames in centered) * 1.05 or 1.0
    for row, (run, frames) in enumerate(zip(validation_runs, centered)):
        for col, xyz in enumerate(frames):
            ax = fig.add_subplot(count, 3, row*3 + col + 1, projection='3d')
            ring = np.vstack([xyz, xyz[0]])
            ax.plot(*ring.T, color='gray', lw=1)
            ax.scatter(*xyz.T, c=np.arange(len(xyz)), cmap='viridis', vmin=0, vmax=len(xyz)-1, s=30)
            ax.set(xlim=(-extent,extent), ylim=(-extent,extent), zlim=(-extent,extent),
                   xlabel='x', ylabel='y', zlabel='z', title=f"{run['name']} — production snapshot {col+1}")
            ax.set_box_aspect((1,1,1))
    plt.show()
    fig, ax = plt.subplots(figsize=(6,4), constrained_layout=True)
    for run in validation_runs:
        coords = run['xyz'][run['production']]
        n_beads = coords.shape[1]
        a, b = np.triu_indices(n_beads, 1)
        genomic = np.minimum(b-a, n_beads-(b-a))
        squared = np.sum((coords[:,a]-coords[:,b])**2, axis=-1)
        s = np.arange(1, n_beads//2+1)
        rms_distance = [np.sqrt(squared[:,genomic==d].mean()) for d in s]
        ax.plot(s*250, rms_distance, 'o-', label=run['name'])
    ax.set(xlabel='Circular genomic separation (kb)', ylabel='R(s) (simulation length units)', title='Spatial distance scaling')
    ax.legend(); ax.grid(alpha=0.25); plt.show()
else:
    print('Structure snapshots and spatial scaling require validation trajectories.')

## 14. Export replica 1 for VMD (PDB + XTC)

This exports the **production segment only** from replica 1, so the trajectory starts after confinement and burn-in. The PDB contains the first production structure and topology; the XTC contains all production frames. Beads are represented as `CA` atoms in generic glycine residues because this is a coarse-grained polymer, not an all-atom protein. A bond is written between each consecutive bead and from the last bead back to the first to represent the ring.

The coordinate arrays are in OpenMiChroM simulation units, which its API reports in nanometers. MDTraj writes the PDB in Å and the XTC in nm. Time uses the saved MD steps and timestep (ps). Install MDTraj in the notebook environment if it is not already available.


In [ ]:
from pathlib import Path
import numpy as np

try:
    import mdtraj as md
except ImportError as exc:
    raise ImportError("This export cell requires MDTraj. Install it with `pip install mdtraj`, then rerun this cell.") from exc

replica_dir = Path('output/structure_validation/replica_1')
trajectory_file = replica_dir / 'trajectory.npz'
config_file = Path('output/structure_validation/config.json')
assert trajectory_file.exists(), f'Missing completed replica 1 trajectory: {trajectory_file}'
assert config_file.exists(), f'Missing validation config: {config_file}'

import json
validation_config = json.loads(config_file.read_text())
with np.load(trajectory_file, allow_pickle=False) as saved:
    xyz_all = saved['xyz']
    phase = saved['phase'].astype(str)
    step = saved['step']

production = phase == 'production'
xyz = np.asarray(xyz_all[production], dtype=np.float32)
steps = step[production]
assert len(xyz) > 0, 'Replica 1 has no saved production frames.'
assert xyz.ndim == 3 and xyz.shape[1] == 20 and xyz.shape[2] == 3
assert np.isfinite(xyz).all(), 'Trajectory contains non-finite coordinates.'

# Build a simple coarse-grained ring topology: one CA bead per GLY residue.
topology = md.Topology()
chain = topology.add_chain('A')
atoms = []
for i in range(xyz.shape[1]):
    residue = topology.add_residue('GLY', chain, resSeq=i + 1)
    atoms.append(topology.add_atom('CA', md.element.carbon, residue))
for i in range(len(atoms)):
    topology.add_bond(atoms[i], atoms[(i + 1) % len(atoms)])

# MDTraj expects coordinates in nm and time in ps.
time_ps = steps.astype(float) * float(validation_config['dt'])
traj = md.Trajectory(xyz=xyz, topology=topology, time=time_ps)
pdb_file = replica_dir / 'replica_1_production.pdb'
xtc_file = replica_dir / 'replica_1_production.xtc'
traj[0].save_pdb(str(pdb_file))
traj.save_xtc(str(xtc_file))
print(f'Wrote {pdb_file} (first production frame)')
print(f'Wrote {xtc_file} ({len(traj)} production frames)')
print('Load both files in VMD; the PDB supplies the matching atom and ring topology.')


## 10. Save your workshop results

Run this before ending a temporary Colab session. The archive includes inputs, trained lambdas, simulated maps, settings, and Adam state. Save your notebook separately with **File → Save** so your plots are retained too.


In [ ]:
import shutil
archive = shutil.make_archive(str(Path.cwd().parent / (Path.cwd().name + '_results')), 'zip', root_dir=Path.cwd())
print('Results archive:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
